---
## Question 3: Diatomic Molecules and Potential Energy Surfaces

Chemistry is about making and breaking bonds. Now that we have a working SCF engine for atoms, we can upgrade it to handle molecules. 

We will model the simplest molecule: **Hydrogen Gas ($H_2$)**. 
When we move to molecules, two major physical things change:
1. **Multi-Center Integrals:** The electrons are now pulled by *two* nuclei across a distance $R$. The integrals become significantly more complex to calculate.
2. **Nuclear Repulsion ($V_{NN}$):** Protons repel each other! Because we use the Born-Oppenheimer approximation (freezing the nuclei in place), we simply calculate the classical Coulomb repulsion between the two nuclei and add it to our final electronic energy: $V_{NN} = \frac{Z_A Z_B}{R}$.

**Your Task:**
1. I have provided a helper function `get_h2_matrices(R, basis)` that instantly returns the integral matrices and the $V_{NN}$ term for a given bond length $R$. It supports a "minimal" basis set and a larger "extended" basis set.
2. Write a `for` loop that iterates over a range of bond lengths from 0.5 to 5.0 Bohrs.
3. Inside the loop, run your SCF engine to find the electronic energy (`E_elec`).
4. Add $V_{NN}$ to `E_elec` to get the **Total Energy**.
5. Do this for *both* the minimal and extended basis sets, and plot both Potential Energy Surfaces (PES) on the same graph!


### The Nightmare of Multi-Center Integrals

Before you write your code, we need to address a massive mathematical elephant in the room: how do we calculate the integrals for a molecule?

In Question 1 and 2, every single Gaussian basis function was centered on the exact same nucleus at the origin $(0,0,0)$. 
But in the $H_2$ molecule, we have two nuclei separated by a distance $R$. Let's call them Nucleus A and Nucleus B. 

If we place one $s$-type orbital on Nucleus A ($\phi_A$) and one on Nucleus B ($\phi_B$), the **Overlap Integral ($S_{AB}$)** becomes a function of the distance between them:
$$ S_{AB} = \int \phi_A(\mathbf{r} - \mathbf{R}_A) \phi_B(\mathbf{r} - \mathbf{R}_B) \, d\mathbf{r} $$

To solve this exactly, computational chemists use the **Gaussian Product Theorem**, which states that the product of two Gaussians centered at different points is just a third Gaussian centered somewhere between them. 
While elegant, coding a true multi-center integral engine requires hundreds of lines of complex calculus and recursion relations. The 2-electron repulsion integrals ($V_{ee}$) become even worse, as they involve four different Gaussians potentially sitting on four different atoms!

$$ (ab|cd) = \iint \phi_A(\mathbf{r}_1) \phi_B(\mathbf{r}_1) \frac{1}{r_{12}} \phi_C(\mathbf{r}_2) \phi_D(\mathbf{r}_2) \, d\mathbf{r}_1 d\mathbf{r}_2 $$

**Our Workaround (The Mock Integrals):**
Because we want you to focus purely on the *architecture* of the SCF algorithm, we are going to bypass the Gaussian Product Theorem. 

Instead of doing the raw 3D calculus, the helper function `get_h2_matrices(R)` uses **analytical mock functions** that perfectly mimic the physical behavior of the real integrals. 
For example, the mock overlap integral is defined as:
$$ S_{AB} \approx e^{-\alpha R} $$
*   As $R \to 0$ (atoms overlap perfectly), $e^0 = 1$.
*   As $R \to \infty$ (atoms are infinitely far apart), $e^{-\infty} = 0$.

These mock functions have the correct asymptotic physics, ensuring your SCF loop behaves exactly like professional software without requiring you to write an industrial-grade integral engine!

In [ ]:
# --- HELPER FUNCTION (DO NOT MODIFY) ---
def get_h2_matrices(R, basis="minimal"):
    """Returns mock integral matrices simulating H2 at distance R."""
    # We use analytical mock functions to simulate the physics of the matrices
    # so you can focus on the SCF algorithm without writing 10,000 lines of calculus!
    
    V_NN = 1.0 / R
    
    if basis == "minimal":
        N = 2
        S = np.array([[1.0, np.exp(-0.8*R)], [np.exp(-0.8*R), 1.0]])
        h = np.array([[-1.0, -np.exp(-R)], [-np.exp(-R), -1.0]])
        V = np.zeros((N,N,N,N))
        V[0,0,0,0] = V[1,1,1,1] = 0.7; V[0,0,1,1] = V[1,1,0,0] = 0.5 / R
        V[0,1,0,1] = V[1,0,1,0] = 0.2 * np.exp(-0.5*R)
        return S, h, V, V_NN
        
    elif basis == "extended":
        N = 4 # Simulating a larger basis set with tight/diffuse functions
        S = np.eye(N) + 0.5 * np.exp(-0.8*R) * np.ones((N,N))
        np.fill_diagonal(S, 1.0)
        h = -1.2 * np.eye(N) - 0.5 * np.exp(-R) * np.ones((N,N))
        V = np.zeros((N,N,N,N))
        for i in range(N):
            for j in range(N):
                V[i,i,j,j] = 0.6; V[i,j,i,j] = 0.3 / R
        return S, h, V, V_NN
# ---------------------------------------

bond_lengths = np.linspace(0.5, 5.0, 50)
energies_minimal = []
energies_extended = []

for R in bond_lengths:
    # --- RUN THE MINIMAL BASIS ---
    S, h_core, V_ee, V_NN = get_h2_matrices(R, basis="minimal")
    N_occ = 1 
    
    # YOUR CODE HERE: Paste your SCF loop. Find E_elec, then add V_NN!
    # E_tot_min = E_elec + V_NN
    # energies_minimal.append(E_tot_min)
    
    # --- RUN THE EXTENDED BASIS ---
    S, h_core, V_ee, V_NN = get_h2_matrices(R, basis="extended")
    
    # YOUR CODE HERE: Paste your SCF loop again for the extended matrices!
    # E_tot_ext = E_elec + V_NN
    # energies_extended.append(E_tot_ext)
    pass

# Plotting the PES
# plt.plot(bond_lengths, energies_minimal, label="Minimal Basis")
# plt.plot(bond_lengths, energies_extended, label="Extended Basis")
# plt.title("H2 Potential Energy Surface")
# plt.xlabel("Bond Length (Bohrs)")
# plt.ylabel("Total Energy (Hartrees)")
# plt.legend()
# plt.show()


---
## Question 3.1: Extracting Physical Properties

Your code just generated a quantum mechanical map of a chemical bond. We can use this curve to extract real physical constants that we can compare to laboratory experiments.

1. **Equilibrium Bond Length ($R_{eq}$):** The distance $R$ at the very bottom of the energy well.
2. **Bond Dissociation Energy ($D_e$):** The energy required to break the bond. This is the difference between the minimum energy and the energy at $R \to \infty$ (where the atoms are completely separated).
3. **Vibrational Force Constant ($k$):** A chemical bond acts like a spring. The stiffness of that spring dictates its vibrational frequency (IR spectroscopy). By taking the second numerical derivative of the energy curve at the minimum ($k = \frac{d^2E}{dR^2}$), we find the force constant.

**Your Task:**
Use `NumPy` array operations to find $R_{eq}$, $D_e$, and $k$ for your **Extended Basis** curve.
*(Hint: Use `np.argmin()` to find the index of the lowest energy. Use `np.gradient()` twice to find the second derivative of the array).*


In [ ]:
# Convert lists to numpy arrays for easier math
# R_array = np.array(bond_lengths)
# E_array = np.array(energies_extended)

# 1. Equilibrium Bond Length
# min_index = ...
# R_eq = R_array[min_index]
# E_min = E_array[min_index]
# print(f"Equilibrium Bond Length: {R_eq:.3f} Bohrs")

# 2. Bond Dissociation Energy (De)
# We approximate R -> infinity using the last point in our array (R = 5.0)
# E_inf = E_array[-1]
# De = E_inf - E_min
# print(f"Bond Dissociation Energy: {De:.4f} Hartrees")

# 3. Vibrational Force Constant (k)
# Find the step size dR
# dR = R_array[1] - R_array[0]
# first_derivative = np.gradient(E_array, dR)
# second_derivative = np.gradient(first_derivative, dR)
# k = second_derivative[min_index]
# print(f"Force Constant (k): {k:.4f} Hartrees/Bohr^2")



### Conceptual Checkpoint: The Dissociation Problem

**Question 3.2:** Look at your plotted Potential Energy Surface. Notice that the Extended Basis set yielded a significantly *lower* energy well than the Minimal Basis set. Does this obey the Variational Theorem? Why does giving the computer more basis functions physically lower the energy?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 3.3 (The RHF Failure):** In the real world, if you stretch an $H_2$ molecule until it breaks ($R \to \infty$), it splits into two neutral Hydrogen atoms ($H^\bullet$ + $H^\bullet$). 

However, we are using Restricted Hartree-Fock (RHF), which mathematically forces the two electrons to share the exact same spatial orbital ($N_{occ}=1$). As the bond stretches to infinity, the math cannot put one electron on the left atom and one on the right atom; it forces both electrons to have a 50% chance of being on the *same* atom, creating a mixture of $H^\bullet + H^\bullet$ and $H^+ + H^-$. 

Because ionic Hydrogen is highly unstable, our RHF energy at $R=5.0$ is actually much *higher* than it should be in reality! What type of Hartree-Fock method would we need to use to allow the two electrons to occupy different spatial orbitals and properly break the bond?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**
